# Deepfake Detection — EfficientNet-B0 Training
**Run all cells top to bottom. Takes ~2-3 hours on T4 for 150K images.**

### Setup required before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Your dataset must be on Google Drive at:
   ```
   /content/drive/MyDrive/deepfake_data/
   ├── REAL/   ← real face images
   └── FAKE/   ← deepfake images
   ```
3. After training, `best_model.pth` is saved to Drive at `/content/drive/MyDrive/deepfake_checkpoints/`
4. Download it and put it in your project's `checkpoints/` folder

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
!pip install -q tqdm torchvision

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/deepfake_data'
SAVE_DIR = '/content/drive/MyDrive/deepfake_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

# Verify data structure
real_dir = os.path.join(DATA_DIR, 'REAL')
fake_dir = os.path.join(DATA_DIR, 'FAKE')
assert os.path.exists(real_dir), f'Missing: {real_dir}'
assert os.path.exists(fake_dir), f'Missing: {fake_dir}'

real_count = len([f for f in os.listdir(real_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
fake_count = len([f for f in os.listdir(fake_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f'REAL images: {real_count:,}')
print(f'FAKE images: {fake_count:,}')
print(f'Total: {real_count + fake_count:,}')

In [ ]:
# ── Cell 4: Full Training Script ───────────────────────────────────────────────
import os, sys, time, json, copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from tqdm.notebook import tqdm
from PIL import Image

# ── Config ────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR  = SAVE_DIR          # Save to Google Drive
BATCH_SIZE      = 64                # T4 has 16GB, 64 fits well
EPOCHS          = 25
LR_HEAD         = 3e-4
LR_FINETUNE     = 3e-5
FREEZE_EPOCHS   = 5
VAL_SPLIT       = 0.15
NUM_WORKERS     = 4
IMG_SIZE        = 224
SEED            = 42
torch.manual_seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Dataset ───────────────────────────────────────────────────────────────────
class SplitableImageFolder(ImageFolder):
    def __getitem__(self, index):
        path, label = self.samples[index]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

def get_dataloaders(data_dir):
    # Load once to get indices
    full = SplitableImageFolder(data_dir, transform=None)
    n = len(full)
    print(f'Total images: {n:,}')
    print(f'Classes: {full.class_to_idx}')

    val_n   = int(n * VAL_SPLIT)
    train_n = n - val_n

    gen = torch.Generator().manual_seed(SEED)
    train_idx, val_idx = random_split(range(n), [train_n, val_n], generator=gen)

    # TWO separate instances with proper transforms (prevents bleed)
    train_ds = SplitableImageFolder(data_dir, transform=train_transforms)
    val_ds   = SplitableImageFolder(data_dir, transform=val_transforms)

    train_subset = Subset(train_ds, train_idx.indices)
    val_subset   = Subset(val_ds,   val_idx.indices)

    labels = [full.targets[i] for i in train_idx.indices]
    n_real = labels.count(full.class_to_idx.get('REAL', 0))
    n_fake = labels.count(full.class_to_idx.get('FAKE', 1))
    print(f'Train: {train_n:,}  (REAL={n_real:,}, FAKE={n_fake:,})')
    print(f'Val  : {val_n:,}')

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    pos_weight = torch.tensor([n_real / (n_fake + 1e-6)])
    return train_loader, val_loader, pos_weight, full.class_to_idx

# ── Model ─────────────────────────────────────────────────────────────────────
def build_model():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = m.classifier[1].in_features  # 1280
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 1),
    )
    return m

# ── Training helpers ──────────────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    bar = tqdm(loader, leave=False, desc='train')
    for imgs, labels in bar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            out  = model(imgs)
            loss = criterion(out, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        pred = (torch.sigmoid(out) > 0.5).float()
        correct += (pred == labels).sum().item()
        total   += labels.size(0)
        bar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.3f}')

    return total_loss / total, correct / total

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False, desc='val'):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device).float().unsqueeze(1)
        out    = model(imgs)
        loss   = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        pred = (torch.sigmoid(out) > 0.5).float()
        correct += (pred == labels).sum().item()
        total   += labels.size(0)
    return total_loss / total, correct / total

# ── Main training ─────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

print('\nLoading dataset...')
train_loader, val_loader, pos_weight, class_map = get_dataloaders(DATA_DIR)
pos_weight = pos_weight.to(device)

model = build_model().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler    = torch.cuda.amp.GradScaler()

# Phase 1: freeze backbone, train head only
for name, p in model.named_parameters():
    if 'classifier' not in name:
        p.requires_grad_(False)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FREEZE_EPOCHS)

best_val_acc = 0.0
best_state   = None
history      = []

print(f'\nPhase 1 — Head warmup ({FREEZE_EPOCHS} epochs, backbone frozen)')
for epoch in range(FREEZE_EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    vl_loss, vl_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    print(f'  Ep {epoch+1:02d}/{FREEZE_EPOCHS} | Train {tr_acc:.3f} ({tr_loss:.4f}) | Val {vl_acc:.3f} ({vl_loss:.4f}) | {time.time()-t0:.0f}s')
    history.append({'epoch': epoch+1, 'phase': 1, 'val_acc': vl_acc, 'val_loss': vl_loss})
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_state   = copy.deepcopy(model.state_dict())
        print(f'  ✓ New best val_acc: {best_val_acc:.4f}')

# Phase 2: full fine-tune with discriminative LRs
print(f'\nPhase 2 — Full fine-tune ({EPOCHS - FREEZE_EPOCHS} epochs)')
for p in model.parameters():
    p.requires_grad_(True)

head_params     = list(model.classifier.parameters())
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': LR_FINETUNE},
    {'params': head_params,     'lr': LR_FINETUNE * 5},
], weight_decay=1e-4)

remaining = EPOCHS - FREEZE_EPOCHS
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining, eta_min=1e-7)

patience, no_improve = 5, 0
for epoch in range(remaining):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    vl_loss, vl_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    print(f'  Ep {FREEZE_EPOCHS+epoch+1:02d}/{EPOCHS} | Train {tr_acc:.3f} ({tr_loss:.4f}) | Val {vl_acc:.3f} ({vl_loss:.4f}) | {time.time()-t0:.0f}s')
    history.append({'epoch': FREEZE_EPOCHS+epoch+1, 'phase': 2, 'val_acc': vl_acc, 'val_loss': vl_loss})

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_state   = copy.deepcopy(model.state_dict())
        no_improve   = 0
        torch.save(best_state, os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
        print(f'  ✓ Saved best — val_acc: {best_val_acc:.4f}')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'  Early stopping after {patience} epochs without improvement')
            break

# Always save final best
if best_state:
    torch.save(best_state, os.path.join(CHECKPOINT_DIR, 'best_model.pth'))

with open(os.path.join(CHECKPOINT_DIR, 'training_history.json'), 'w') as f:
    json.dump({'best_val_acc': best_val_acc, 'class_map': class_map, 'history': history}, f, indent=2)

print(f'\n{"="*60}')
print(f'Training complete!')
print(f'Best val accuracy: {best_val_acc*100:.2f}%')
print(f'Saved to: {CHECKPOINT_DIR}/best_model.pth')
print(f'{"="*60}')

In [ ]:
# ── Cell 5: Sanity check — test the saved model ────────────────────────────────
# Load best model and run a quick inference check
import torch, os
from torchvision import models, transforms
import torch.nn as nn
from PIL import Image
import requests
from io import BytesIO

def build_model():
    m = models.efficientnet_b0(weights=None)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 1),
    )
    return m

model_path = os.path.join(SAVE_DIR, 'best_model.pth')
assert os.path.exists(model_path), f'Model not found at {model_path}'

model = build_model()
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Test on a random real photo from the web
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/320px-Gatto_europeo4.jpg'
img = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
tensor = val_tf(img).unsqueeze(0)

with torch.no_grad():
    logit = model(tensor).item()
    prob  = torch.sigmoid(torch.tensor(logit)).item()

print(f'Raw logit: {logit:.4f}')
print(f'Sigmoid:   {prob:.4f}')
print()

# CRITICAL: if prob is stuck near 0.48-0.52 for ANY input, the model is collapsed
if abs(prob - 0.5) < 0.05:
    print('WARNING: Model outputs near 0.5 — likely collapsed. Training may not have converged.')
    print('Check training_history.json to see if val_acc improved above 0.55 within first 3 epochs.')
else:
    print('Model is responding to input (not collapsed).')
    print(f'Note: class_map tells you which class = 0/1. Check training_history.json for class_map.')

In [ ]:
# ── Cell 6: Plot training curves ───────────────────────────────────────────────
import json, os
import matplotlib.pyplot as plt

with open(os.path.join(SAVE_DIR, 'training_history.json')) as f:
    data = json.load(f)

history = data['history']
epochs   = [h['epoch'] for h in history]
val_acc  = [h['val_acc'] for h in history]
val_loss = [h['val_loss'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, val_acc, 'b-o', markersize=4)
ax1.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='random baseline')
ax1.set_title('Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, val_loss, 'r-o', markersize=4)
ax2.set_title('Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.grid(True)

plt.suptitle(f"Best Val Acc: {data['best_val_acc']*100:.2f}%  |  Class map: {data['class_map']}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=120)
plt.show()
print(f"Saved curves to {SAVE_DIR}/training_curves.png")

## After training — integrate with your local project

1. **Download** `best_model.pth` from your Drive (`deepfake_checkpoints/best_model.pth`)
2. **Place it** at `checkpoints/best_model.pth` in your project folder
3. **Note the `class_map`** from Cell 6's title — it shows whether FAKE=0 or FAKE=1
   - If `class_map = {'FAKE': 0, 'REAL': 1}` then the model predicts REAL as the positive class
   - Check `backend/inference.py` and adjust the sigmoid interpretation if needed
4. **Restart backend**: `./start.sh`

### What good training looks like
- Phase 1 (epoch 1-5): val_acc should climb from ~0.50 to ~0.75 within 2-3 epochs
- Phase 2 (epoch 6-25): val_acc should stabilize at 0.90-0.97
- If epoch 1 val_acc is stuck at 0.48-0.52 → data structure problem (check REAL/ FAKE/ folders)
- If accuracy plateaus at 0.70-0.75 → run more epochs or reduce LR_FINETUNE